In [ ]:
import os, glob

import cytoflow as flow
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from flow_analysis_tools import flow_violin
from flow_analysis_tools.color_palettes import *

## Notebook and Experiment Setup

In [ ]:
%matplotlib widget
matplotlib.rc('figure', dpi = 160)
plt.style.use('ggplot')

In [ ]:
# DATA_DIR = ""
DATA_DIR = '/Users/alexandresathler/Library/CloudStorage/Box-Box/HsiungLab/arsathler/Flow Results/2602-LX2-CRch-Const-Rep2/260225_LX2_CRch_Const_CD81_r2d14'
FCS_FILES = glob.glob(os.path.join(DATA_DIR, "*.fcs"))
YLIMITS = (0, 7)
print('\n'.join([f"'{x}'" for x in FCS_FILES]))

## Import Setup

In [ ]:
CONSTRUCT_NAME_N_UNDERSCORE = 2

construct_names = {
    "zim3": "dCas9-ZIM3",
    "ZIM3": "dCas9-ZIM3",
    "ZIM3-sgNT": "dCas9-ZIM3",
    "ZIM3-sgCD81": "dCas9-ZIM3",
    "pCH95": "H3t-D3L-dCas9",
    "pCH95-sgNT": "H3t-D3L-dCas9",
    "pCH95-sgCD81": "H3t-D3L-dCas9",
    "pCH96": "H3t-D3L-dCas9-KOX1",
    "pCH96-sgNT": "H3t-D3L-dCas9-KOX1",
    "pCH96-sgCD81": "H3t-D3L-dCas9-KOX1",
    "pCH97": "H3t-KOX1-D3L-dCas9",
    "pCH97-sgNT": "H3t-KOX1-D3L-dCas9",
    "pCH97-sgCD81": "H3t-KOX1-D3L-dCas9",
    "sgNT-1X": "1X",
    "sgNT-20X": "20X",
    "HEK-CLTA": "HEK",
    "sgCLTA-1X": "1X",
    "sgCLTA-20X": "20X",
    "sgCol1A1-1X": "1X",
    "sgCol1A1-20X": "20X",
    "LX2-TGFb-Ab": "LX-2 (TGFß)",
    "LX2-TGFb-nAb": "LX-2 (TGFß)",
    "LX2-Ab": "LX-2",
    "LX2-nAb": "LX-2"
}

def get_construct_name(name):
    this_name = name.split("_")[-CONSTRUCT_NAME_N_UNDERSCORE]
    if this_name in construct_names.keys():
        return construct_names[this_name]
    return this_name

def get_guide_type(name):
    # if '_NT_' in name or '_nt_' in name:
    #     return "non-targeting"
    # else:
    #     return "targeting"
    if "_HEK_" in name: return "CLTA-"
    elif "_HEK-CLTA_" in name: return "CLTA+"
    elif "sgCLTA" in name: return "sgCLTA"
    elif "sgCol1A1" in name: return "sgCol1A1"
    elif "sgNT" in name: return "sgNT"
    elif "sgCD81" in name: return "sgCD81"
    elif "-Ab" in name: return "PE-αCD81"
    elif "-nAb" in name: return "No Ab"

In [ ]:
CONSTRUCT_NAMES = sorted([get_construct_name(fcs_file) for fcs_file in FCS_FILES])

import_op = flow.ImportOp(
    conditions = {
        "Construct": "category",
        "Guide": "category"
    },
    tubes = [flow.Tube(
        file=fcs_file, 
        conditions={
            "Construct": get_construct_name(fcs_file),
            "Guide": get_guide_type(fcs_file),
    }) for fcs_file in FCS_FILES],
    # channels = channel_names
)

ex = import_op.apply()

## Select Cells

In [ ]:
%matplotlib widget

In [ ]:
single_gate = flow.PolygonOp(
    name="Single_Cell",
    xchannel="FSC-A",
    ychannel="SSC-A"
)

single_gate.default_view(
    density = True,
    huescale = "log",
    interactive = True
).plot(ex, gridsize = 100)

In [ ]:
ex = single_gate.apply(ex)
plt.close()

## Gating

### Gating Template
Use the template below for help with gating

```
quad_gate = flow.QuadOp(
    name="",
    xchannel="",
    ychannel="",
)
single_gate = flow.PolygonOp(
    name="",
    xchannel="",
    ychannel="",
    yscale="logicle"
)
___gate = flow.PolygonOp(name="") # placeholder variable
___gate.default_view(
    density = True,
    huescale = "log",
    interactive = True,
    subset = "Single_Cell == True"
) #.plot(ex_single, gridsize=100)
ex_single_gated = ___gate.apply(ex_single)
plt.close()
```

In [ ]:
quad_gate = flow.QuadOp(
    name="d_pos",
    xchannel="BL1-A",
    ychannel="VL1-A",
    # xscale="logicle",
    # yscale="logicle"
)

quad_gate.default_view(
    density = True,
    huescale = "log",
    interactive = True,
    subset = "Single_Cell == True",
    xscale="logicle", yscale="logicle"
).plot(ex, gridsize=100)

In [ ]:
ex = quad_gate.apply(ex)
plt.close()

In [ ]:
ex.data.to_csv(os.path.join(DATA_DIR, "processed_data.csv"))

## Graphing

In [ ]:
flow.DensityView(
    xchannel="FSC-A",
    ychannel="RL1-A",
    yscale="log",
    huescale="log",
    subset="Single_Cell == True"
).plot(ex)

In [ ]:
flow.DensityView(
    xchannel="VL1-A",
    ychannel="RL1-A",
    yscale="log",
    xscale="log",
    # huescale="log",
    subset="Single_Cell == True"
).plot(ex)

In [ ]:
%matplotlib inline
plt.close()
plt.close()

In [ ]:
plot_data = ex.data[ex.data["Single_Cell"]==True]

In [ ]:
order = ['dCas9-ZIM3', 'H3t-D3L-dCas9', 'H3t-D3L-dCas9-KOX1', 'H3t-KOX1-D3L-dCas9']
order_no_krab = ['H3t-D3L-dCas9', 'H3t-D3L-dCas9-KOX1', 'H3t-KOX1-D3L-dCas9']

In [ ]:
channel_labels = {
    "Construct (BFP)": "VL1-A",
    "Guide (GFP)": "BL1-A",
    "RFP": "RL1-A",
    "CD81 (PE)": "YL1-A"
}

channel_cmaps = {
    "Construct (BFP)": black_blue_pastel_palette,
    "Guide (GFP)": black_green_pastel_palette,
    "RFP": black_red_pastel_palette,
    "CD81 (PE)": black_yellow_pastel_palette
}

for fluor, flow_channel in channel_labels.items():
    # See: https://matplotlib.org/stable/gallery/lines_bars_and_markers/scatter_hist.html#id1
    fig, axs = plt.subplot_mosaic(
        [
            ["Mock", "Construct"]
        ],
        figsize=(8,4),
        width_ratios=(1,3),
        sharey=True
        # layout="consrained"
    )
    flow_violin(
        data=plot_data[(plot_data["Construct"] == "LX-2")],
        x="Construct",
        y=flow_channel,
        hue="Guide",
        palette=channel_cmaps[fluor],
        split=True,
        log_y_axis=True,
        hue_order=["PE-αCD81", "No Ab"],
        order=["LX-2"],
        ylabel=fluor,
        xlabel="Control",
        ylim=YLIMITS,
        ax=axs['Mock']
    )

    flow_violin(
        data=plot_data[plot_data["Construct"] != "LX-2"],
        x="Construct",
        y=flow_channel,
        hue="Guide",
        palette=channel_cmaps[fluor],
        order=order,
        split=True,
        log_y_axis=True,
        xlabel="Epigenetic Construct (Constitutive)",
        hue_order=["sgNT", "sgCD81"],
        legend_labels=["sgNT", "sgCD81"],
        ylim=YLIMITS,
        ax=axs['Construct']
    )
    axs['Mock'].legend(title="LX-2", loc="upper right")
    axs['Construct'].legend(title="Guide", loc="upper right")
    axs['Construct'].set_xticklabels(axs['Construct'].get_xticklabels(), rotation=12.25)
    plt.savefig(os.path.join(DATA_DIR, f"{fluor}.png"))

In [ ]:
plot_data.d_pos.unique()

In [ ]:
# See: https://matplotlib.org/stable/gallery/lines_bars_and_markers/scatter_hist.html#id1
fig, axs = plt.subplot_mosaic(
    [
        ["Mock", "Construct"]
    ],
    figsize=(8,4),
    width_ratios=(1,3),
    sharey=True
    # layout="consrained"
)

flow_violin(
    data=plot_data[(plot_data["Construct"] == "LX-2")],
    x="Construct",
    y="YL1-A",
    hue="Guide",
    palette=black_yellow_pastel_palette,
    # order=list(CONSTRUCT_NAMES),
    split=True,
    log_y_axis=True,
    hue_order=["PE-αCD81", "No Ab"],
    order=["LX-2"],
    ylabel=fluor,
    xlabel="Control",
    ylim=YLIMITS,
    ax=axs['Mock']
)

flow_violin(
    data=plot_data[(plot_data["d_pos"] == "d_pos_2") & (plot_data["Construct"] != "LX-2")],
    x="Construct",
    y="YL1-A",
    hue="Guide",
    palette=black_yellow_pastel_palette,
    order=order_no_krab,
    split=True,
    log_y_axis=True,
    xlabel="Epigenetic Construct (Constitutive)",
        hue_order=["sgNT", "sgCD81"],
        legend_labels=["sgNT", "sgCD81"],
    ylim=YLIMITS,
    ax=axs['Construct']
)
axs['Mock'].legend(title="LX-2", loc="upper right")
axs['Construct'].legend(title="Guide", loc="upper right")
axs['Construct'].set_xticklabels(axs['Construct'].get_xticklabels(), rotation=12.25)
plt.savefig(os.path.join(DATA_DIR, "yfp_construct+guide+_only.png"))